In [ ]:
# einsum chained contraction performance test. einsum apparently has some problems regarding chained contraction, as it doesn't automatically know which intermediate matrices to "collapse". For example, when chaining multiplications with A, usually and intermediate matrix AA is calculated and then multiplied with the next A. when using the naive chained einsum with ij, jk, kl -> lm for three A's, it will probably broadcast ALL dims, do many multiplications and then sum them at the end

import numpy as np

mat_size = 10
A = np.random.rand(mat_size, mat_size)

print("dot product contraction")
%timeit A.dot(A).dot(A).dot(A)

print("standard einsum, no optimization")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A)

print("directly using optimization flag")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=True)

print("pre-storing optimization")
opt_path = np.einsum_path("ij, jk, kl, lm -> im", A, A, A, A, optimize=False)[0] # remove info string!
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=opt_path)

print("directly specifiying path")
%timeit np.einsum("ij, jk, kl, lm -> im", A, A, A, A, optimize=["einsum_path", (0, 1), (0, 1), (0, 1)])

print("taking operations apart")
def chained_einsum():
    a1 = np.einsum("ij, jk -> ik", A, A)
    a2 = np.einsum("ij, jk -> ik", a1, A)
    a3 = np.einsum("ij, jk -> ik", a2, A)
%timeit chained_einsum()

print("done!")

dot product contraction
3.75 μs ± 148 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
standard einsum, no optimization
714 μs ± 12.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
directly using optimization flag
169 μs ± 4.76 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
pre-storing optimization
769 μs ± 51.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
directly specifiying path
121 μs ± 3.15 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
taking operations apart
14.1 μs ± 417 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
done!


In [11]:
import numpy as np

mat_size = 100
batch_size = 10
A = np.random.rand(batch_size, batch_size, mat_size, mat_size)

print("einsum matmul")
%timeit np.einsum("ABij, ABjk -> ABik", A, A)

print("numpy @")
%timeit A @ A

print("numpy matmul")
%timeit np.matmul(A, A)


einsum matmul
28.8 ms ± 475 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
numpy @
8.4 ms ± 27.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
numpy matmul
8.12 ms ± 83.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
